# AI AppCosts

**Module 13 · Lesson 13.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Tokens, Not Requests
- The Cost of One Request
- The Hidden Multipliers
- The Three Big Discounts
- Cost at Scale and Unit Economics
- Cost Attribution: Meter Every Call

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## The invoice shock: a “simple chatbot” that cost a fortune

> 💡 **Info**
>
> The chatbot worked. Users loved it. Then the first invoice arrived, and it was **an order of magnitude** higher than you had modeled. Nothing broke; the bill was exactly right. The problem was your mental model: you priced it like every SaaS you had ever built - a flat rate per request, ten thousand calls for a predictable sum. But an LLM does not bill per request. It bills **per token**, input and output priced separately, and the tokens that bury you are the ones you never see: the **four thousand words of retrieved context** stapled onto a fifty-word question, the **hidden reasoning tokens** behind a one-line answer, the **five model calls** your agent quietly makes for every user turn. A “simple chatbot” that felt like it should cost a fraction of a cent per chat turns out to cost real money per conversation, and at scale that is the difference between a healthy margin and a feature you cannot afford to run. This lesson opens Module 13 by building the cost model and the tooling to know every token before you spend it: the per-token math, the hidden multipliers, the three discounts that stack, unit economics and gross margin, per-user attribution, and a dashboard that names your single biggest cost driver.

### 🎯 What you will be able to do after this lesson

- **Build** AI cost tooling — a per-token calculator, a hidden-multiplier estimator, a discount-stacker, a unit-economics projector, a per-call cost meter, and a spend dashboard — as runnable models plus an illustrative metering wrapper.

- **Compare** per-request SaaS pricing with per-token LLM pricing, and a naive estimate with the real (usage-based) cost.

- **Operate** the three cost levers (cached input, batch, model tiering), per-user/feature attribution, and a median-vs-p99 read on agent-run cost.

- **Evaluate** an AI feature’s cost: do you know its cost per request, per user, its gross margin, and its single biggest driver?

> 📦 **Info**
>
> ✅ Before you startIn **12.4** you cached a repeated prefix to skip recompute — that same cache is the biggest discount here (about ninety percent off those input tokens). In **12.8** cost per request was one of the golden signals, and the trace each call rides on is where you attach the cost metadata to attribute spend. In **Module 8** retrieval stuffed thousands of context tokens into the prompt — those are input tokens you pay for on every call. Deeper optimization (model routing, self-hosting, GPU economics) comes later in Module 13; org-level cost governance is in Module 14.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🚖 **Analogy**
>
> AI pricing is a **taxi meter, not a flat bus fare**. A bus charges the same whether you ride one stop or twenty (per-request SaaS: ten thousand calls, one predictable sum). A taxi charges by *distance and time* - a short hop is cheap, a cross-town crawl in traffic is not (per-token LLM: a tiny query is cheap, a huge context with a long answer is not). And the meter keeps running for things you did not ask for: the long way round the model takes to ‘think’, the extra legs an agent drives. **Where it breaks down:** a taxi meter is visible on the dash the whole ride, while the LLM meter is invisible until the invoice - which is exactly why you build one yourself.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“LLM cost is a flat rate per call, like SaaS.”** It is per token, input and output priced separately, output costing more - so a long answer or a big context costs far more than a short one.
> - **“I only pay for what the user typed.”** The system prompt, the retrieved RAG context, the conversation history, and hidden reasoning tokens are all billable - and usually dwarf the user’s question.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that quietly drain the budget. **Running a frontier model on every trivial query** — the same easy task can be an order of magnitude cheaper on a small model; tier by difficulty. And **trusting the average cost per user** — spend is heavily skewed, so a handful of users or runaway agent runs can eat all your margin while the mean looks perfectly healthy. Watch the tail (p99), not just the average.

---

## 🎯 Concept 1: Tokens, Not Requests

### Tokens, Not Requests

Traditional SaaS bills a flat rate per request. An LLM bills per token, with input and output priced separately and output costing more. So a short query and a huge document cost wildly differently on the same model, and the same task can cost ~600x across the model range.

Start with the single fact that resets your intuition: **cost is tokens times price, not calls times price**. A traditional SaaS bills a flat rate per request — ten thousand calls cost the same predictable amount whether each does a little or a lot. An LLM bills **per token**, and two properties make that wild. First, **input and output are priced separately, and output costs more** (roughly two to five times, because each output token is a full forward pass through the model). Second, model prices span an enormous range across providers like **OpenAI**, **Anthropic**, and **Google** — the cheapest small models to the priciest frontier and reasoning tiers cover about a **six-hundred-fold** spread. Put together: a one-hundred-token query and a ten-thousand-token document cost a hundred times differently on the *same* model (their input alone), and the *same* request can cost a hundred times more on a frontier model than on a small one. A “simple chatbot” is never one price; it is a meter. The block contrasts flat SaaS with per-token cost, keyless (illustrative prices).

> 🚙 **Analogy**
>
> It is a **freight bill, not a postage stamp**. A stamp is a flat rate: the letter goes for the same price whether the envelope is nearly empty or stuffed full. Freight is by *weight and distance*: a light parcel across town is cheap, a heavy one across the country is not. LLM tokens are freight — a short prompt with a short answer is a light parcel, but a huge retrieved context with a long, reasoned answer is a container ship, and you pay by what is actually in the hold, not by the fact that you sent ‘one shipment’.

The same 100-token query is sent to a small model and to a frontier model. Roughly how much more does the frontier model cost for that one request?

**📝 `01_tokens_not_requests.py`** - *runnable*

In [ ]:
# ILLUSTRATIVE prices (USD per 1M tokens; check provider docs for live numbers).
# Two facts: input and output are priced separately, and output costs more.
PRICES = {  # tier: (input_per_1M, output_per_1M)
    "flash":    (0.15,  0.60),   # small / fast
    "frontier": (3.00, 15.00),   # top-tier (output is 5x input)
}
def cost(tier, in_tok, out_tok):
    pin, pout = PRICES[tier]
    return (in_tok * pin + out_tok * pout) / 1_000_000

# Traditional SaaS: a flat rate per REQUEST.
print("Traditional SaaS: $0.01 per call -> 10,000 calls = ${:.0f}. Simple and predictable.".format(10000 * 0.01))
print()
print("LLM: cost = tokens x price. A 100-token query vs a 10,000-token document, same model (flash):")
a = cost("flash", 100, 200)
b = cost("flash", 10000, 200)
print("  query    (in 100,   out 200): ${:.6f}".format(a))
print("  document (in 10000, out 200): ${:.6f}   <- its INPUT alone costs 100x the query's input".format(b))
print()
print("Same request (in 100, out 200) across model tiers:")
for tier in ["flash", "frontier"]:
    print("  {:<9} ${:.6f}".format(tier, cost(tier, 100, 200)))
print("  frontier is {:.0f}x the flash cost - and the full market spans a ~600x range.".format(cost("frontier",100,200)/cost("flash",100,200)))
print()
print("Cost is tokens x price, NOT calls x price. A long answer or a big context costs far more.")

# Output:
# Traditional SaaS: $0.01 per call -> 10,000 calls = $100. Simple and predictable.
#
# LLM: cost = tokens x price. A 100-token query vs a 10,000-token document, same model (flash):
#   query    (in 100,   out 200): $0.000135
#   document (in 10000, out 200): $0.001620   <- its INPUT alone costs 100x the query's input
#
# Same request (in 100, out 200) across model tiers:
#   flash     $0.000135
#   frontier  $0.003300
#   frontier is 24x the flash cost - and the full market spans a ~600x range.
#
# Cost is tokens x price, NOT calls x price. A long answer or a big context costs far more.

- Traditional SaaS is a **flat rate per request** — predictable, and independent of how much work each call does.
- An LLM charges **tokens times price**: the ten-thousand-token document’s *input alone* costs a hundred times the hundred-token query’s input.
- The **same request** costs many times more on a frontier tier than on flash — and the full market spans about a six-hundred-fold range.
- The takeaway that reframes everything: cost is tokens times price, not calls times price — a long answer or a big context costs far more.

#### 💡 What just happened

⚡ What just happened? An LLM bills per token, with input and output priced separately and output costing more, across a ~600x model range - so cost is tokens x price, not calls x price. Tradeoff / the framing for the module: every later step is a way to see, and shrink, that token bill. A short query is cheap; a huge context with a long reasoned answer is not.

- A flat SaaS meter next to a per-token meter
- Drag the query size (100 to 10000 tokens) and switch the model tier, and watch the two diverge

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Cost of One Request

### The Cost of One Request

One request costs input tokens x input price + output tokens x output price. Output can dominate even with fewer tokens because it is priced higher. And the input you did not write - a RAG call’s retrieved context - is usually the bulk of the input you pay for.

Now price a single request exactly. The cost is `input_tokens x input_price + output_tokens x output_price`. Two things surprise people. First, the **output can cost more than the input even with fewer output tokens**, because output is priced higher — a concise answer is not automatically cheap. Second, and the bigger trap, is the **input you did not write**. When you add retrieval (Module 8), a tiny user question rides into the model with *thousands* of tokens of retrieved context, plus the system prompt and the conversation history. So a fifty-token question can carry four thousand tokens of context — the context is about **ninety-nine percent** of the input you pay for, on *every* call. The rule: count the **whole prompt** — system prompt plus retrieved context plus history plus query — not just what the user typed. The block prices a chat request and a RAG request, keyless.

> 📦 **Analogy**
>
> It is a **printed page plus the packet stapled behind it**. You write a one-line question at the top (the query), but before it goes to the model you staple on the entire reference packet it needs to answer — the manual, the past conversation, the instructions (the system prompt and retrieved context). The courier weighs the *whole* stack, not your one line. People budget for the sentence they wrote and are shocked by the ream stapled behind it — that packet is the real weight of every RAG call.

A RAG request has a 50-token question and 4,000 tokens of retrieved context. Which drives the input cost?

**📝 `02_one_request.py`** - *runnable*

In [ ]:
# The cost of ONE request = input cost + output cost. Output is priced higher, so it can dominate.
PRICES = {"flash": (0.15, 0.60), "frontier": (3.00, 15.00)}
def parts(tier, in_tok, out_tok):
    pin, pout = PRICES[tier]
    ic = in_tok * pin / 1_000_000
    oc = out_tok * pout / 1_000_000
    return ic, oc, ic + oc

ic, oc, total = parts("flash", 500, 300)
print("A chat request on flash (in 500, out 300):")
print("  input cost  ${:.6f}   (500 tokens x $0.15/M)".format(ic))
print("  output cost ${:.6f}   (300 tokens x $0.60/M)".format(oc))
print("  total       ${:.6f}   <- output cost > input cost despite FEWER output tokens (output is 4x)".format(total))
print()
# RAG: the input you did NOT write. A tiny query rides with a big retrieved context.
query, context, out = 50, 4000, 400
ic2, oc2, total2 = parts("flash", query + context, out)
print("A RAG request on flash (query {}, retrieved context {}, out {}):".format(query, context, out))
print("  total input tokens: {} (query {} + context {})".format(query + context, query, context))
print("  context is {:.0f}% of the input - the 'hidden' cost you pay on EVERY call".format(context / (query + context) * 100))
print("  total cost ${:.6f}   (you thought you sent {} tokens; you sent {})".format(total2, query, query + context))
print()
print("Count the WHOLE prompt: system prompt + retrieved context + history + query, not just what the user typed.")

# Output:
# A chat request on flash (in 500, out 300):
#   input cost  $0.000075   (500 tokens x $0.15/M)
#   output cost $0.000180   (300 tokens x $0.60/M)
#   total       $0.000255   <- output cost > input cost despite FEWER output tokens (output is 4x)
#
# A RAG request on flash (query 50, retrieved context 4000, out 400):
#   total input tokens: 4050 (query 50 + context 4000)
#   context is 99% of the input - the 'hidden' cost you pay on EVERY call
#   total cost $0.000847   (you thought you sent 50 tokens; you sent 4050)
#
# Count the WHOLE prompt: system prompt + retrieved context + history + query, not just what the user typed.

- A chat request’s **output cost exceeds its input cost** even with fewer output tokens, because output is priced higher (four times, here).
- A RAG request’s input is **query plus retrieved context**; the context is about ninety-nine percent of the input tokens.
- You thought you sent fifty tokens; you sent four thousand — the **hidden cost of RAG** you pay on every call.
- The rule: count the whole prompt (system prompt + context + history + query), not just what the user typed.

#### 💡 What just happened

⚡ What just happened? One request costs input x input-price + output x output-price; output can dominate despite fewer tokens, and a RAG call’s retrieved context is usually ~99 percent of the input you pay for. Tradeoff: richer context and longer answers cost real money per call, so trim the prompt and cap the output when you can. Next: the costs that do not even show up in the visible request.

- Input and output cost bars for one request
- Add a RAG context block and watch the input bar balloon while the query stays tiny (context ~99 percent)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Hidden Multipliers

### The Hidden Multipliers

Your bill is often several times a naive estimate, because of tokens you never see: reasoning models bill a long internal chain-of-thought as output, and an agent fans one user request into many LLM calls. Estimate from the actual token usage, not the visible text.

Even after counting the whole prompt, the real bill is often several times what you would guess — because of tokens that are billed but invisible. The first multiplier is **reasoning tokens**. A reasoning model produces a long internal chain-of-thought before its answer, and that reasoning is **billed as output tokens but never shown to you**. A one-line answer with five hundred visible output tokens might have consumed three thousand hidden reasoning tokens — so the output you are billed for is *seven times* the output you can see. The second multiplier is the **agent loop**: one user request fans out into many LLM calls (plan, call a tool, call another, reflect, answer), so an agent multiplies the per-request cost by its number of steps. The third is the **RAG context** from Step 2. The lesson: **estimate from the actual token usage** the API reports, never from the visible text — the visible answer is the tip of the iceberg. The block shows both multipliers, keyless.

> 🧊 **Analogy**
>
> It is the **iceberg under the answer**. What the user sees — the tidy one-paragraph reply — is the tip above the waterline. Below it sits the mass you still paid to move: the model’s long private deliberation (reasoning tokens), and, for an agent, the several round-trips it made to tools and back before it surfaced an answer. Budget from the tip and you hit the hull; budget from the *whole* iceberg — the usage the API actually reports — and you know the real displacement.

A reasoning model returns a 500-token answer but used 3,000 hidden reasoning tokens. What output are you billed for?

**📝 `03_hidden_multipliers.py`** - *runnable*

In [ ]:
# The bill is often several times a naive estimate, because of tokens you never see.
PRICES = {"frontier": (3.00, 15.00)}
def cost(in_tok, out_tok):
    pin, pout = PRICES["frontier"]
    return (in_tok * pin + out_tok * pout) / 1_000_000

# (1) REASONING tokens: billed as output, but hidden from you.
visible_out, reasoning, in_tok = 500, 3000, 1000
naive = cost(in_tok, visible_out)                 # what you'd estimate from the visible answer
real = cost(in_tok, visible_out + reasoning)      # what you're actually billed
print("Reasoning model - a one-line answer:")
print("  visible output tokens:  {}".format(visible_out))
print("  hidden reasoning tokens: {}  (billed as output, never shown)".format(reasoning))
print("  billed output = {} -> {}x the visible output".format(visible_out + reasoning, (visible_out + reasoning) // visible_out))
print("  naive estimate ${:.6f}   real cost ${:.6f}   = {:.1f}x more than you'd guess".format(naive, real, real / naive))
print()
# (2) AGENT loops: one user request fans out into many LLM calls.
calls = [("plan", 800, 200), ("tool call", 900, 150), ("tool call", 950, 150), ("reflect", 1100, 200), ("answer", 1200, 400)]
print("Agent - one user request fans out into {} LLM calls:".format(len(calls)))
run_total = 0.0
for name, i, o in calls:
    c = cost(i, o); run_total += c
    print("  {:<10} in {:>4} out {:>3}  ${:.6f}".format(name, i, o, c))
single = cost(800, 200)
print("  one user request costs ${:.6f} = {:.1f}x a single call - the agent multiplier".format(run_total, run_total / single))
print()
print("Estimate from the ACTUAL token usage (reasoning + all calls), never from the visible text.")

# Output:
# Reasoning model - a one-line answer:
#   visible output tokens:  500
#   hidden reasoning tokens: 3000  (billed as output, never shown)
#   billed output = 3500 -> 7x the visible output
#   naive estimate $0.010500   real cost $0.055500   = 5.3x more than you'd guess
#
# Agent - one user request fans out into 5 LLM calls:
#   plan       in  800 out 200  $0.005400
#   tool call  in  900 out 150  $0.004950
#   tool call  in  950 out 150  $0.005100
#   reflect    in 1100 out 200  $0.006300
#   answer     in 1200 out 400  $0.009600
#   one user request costs $0.031350 = 5.8x a single call - the agent multiplier
#
# Estimate from the ACTUAL token usage (reasoning + all calls), never from the visible text.

- **Reasoning tokens**: a five-hundred-token visible answer carried three thousand hidden reasoning tokens — the billed output is seven times the visible output.
- The naive estimate (from the visible answer) is several times too low; the **real cost** comes from the full usage.
- **Agent loops**: one user request fans into five LLM calls, so the cost is several times a single call — the agent multiplier.
- The rule: estimate from the **actual token usage** (reasoning + every call), never from the visible text.

#### 💡 What just happened

⚡ What just happened? The bill is often several times a naive estimate because of tokens you never see: reasoning billed as hidden output, and an agent’s many calls per request. Tradeoff: reasoning models and agents buy quality and autonomy at a real, easy-to-underestimate token cost - so estimate from usage. Next: the levers that cut all of this.

- Peel back the visible answer to reveal the reasoning tokens underneath (500 visible + 3000 hidden)
- Fan one request into an agent’s five calls and watch the cost multiply

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: The Three Big Discounts

### The Three Big Discounts

Three levers cut the bill hard and they stack: prompt caching (a reusable prefix at ~10 percent of input price), the batch API (~50 percent off async work), and model tiering (route easy queries to a small model). Caching plus batch stacks to roughly a quarter of the standard bill.

The bill is not fixed — three levers cut it hard, and two of them **stack**. First, **prompt caching** (the 12.4 mechanic): a reusable prefix, like a long system prompt or a stable context, can be cached so subsequent reads cost roughly **a tenth** of the input price — up to about ninety percent off those cached tokens. Second, the **batch API**: asynchronous, non-urgent work (nightly summaries, bulk classification) gets about **fifty percent off** all tokens for accepting a slower turnaround. Third, **model tiering**: route the easy, short queries to a small model and reserve the frontier model for the genuinely hard ones — often an order-of-magnitude saving with no quality loss on the easy path. The two discounts compound: caching the reusable prefix *and* batching the async work brings the effective per-call cost to roughly **a quarter** of the standard rate. The block stacks the discounts on one request, keyless.

> 🎟️ **Analogy**
>
> It is a **season pass, a bulk rate, and the right seat**. The season pass is prompt caching: pay once for the part you reuse every visit (the system prompt) instead of full price each time. The bulk rate is the batch API: accept a slower, scheduled delivery and the whole order is discounted. And the right seat is model tiering: you do not fly first class to the corner shop — match the model to the trip. Stack the pass and the bulk rate and you are paying a fraction of the walk-up price for the same journey.

You cache a reusable system prompt AND batch the async work. Roughly what fraction of the standard bill do you pay?

**📝 `04_three_discounts.py`** - *runnable*

In [ ]:
# Three levers cut the bill hard, and they STACK. (ILLUSTRATIVE rates.)
IN, OUT = 3.00, 15.00           # frontier $/1M
CACHED_IN = IN * 0.10           # cached input ~10% of input price (up to ~90% off)
BATCH = 0.50                    # batch API ~50% off everything

# A request with a big reusable prefix (a stable system prompt / context) + a little fresh input.
cacheable, fresh_in, out = 3500, 500, 400
def total(cached, batch):
    if cached:
        in_cost = (cacheable * CACHED_IN + fresh_in * IN) / 1_000_000
    else:
        in_cost = (cacheable + fresh_in) * IN / 1_000_000
    c = in_cost + out * OUT / 1_000_000
    return c * (BATCH if batch else 1.0)

std = total(False, False)
print("A request (reusable prefix {} + fresh input {} + output {}) on frontier:".format(cacheable, fresh_in, out))
print("  standard                 ${:.6f}".format(std))
print("  + prompt caching         ${:.6f}   ({:.0f}% of standard)".format(total(True, False),  total(True, False)  / std * 100))
print("  + batch (async)          ${:.6f}   ({:.0f}% of standard)".format(total(False, True),  total(False, True)  / std * 100))
print("  + caching AND batch      ${:.6f}   ({:.0f}% of standard)  <- the discounts stack".format(total(True, True), total(True, True) / std * 100))
print()
# The third lever: model tiering (route the easy queries down).
flash = (500 * 0.15 + 400 * 0.60) / 1_000_000
front = (500 * 3.00 + 400 * 15.00) / 1_000_000
print("Model tiering - the same short query on flash vs frontier:")
print("  flash ${:.6f}  vs  frontier ${:.6f}  = {:.0f}x cheaper (if the small model is good enough)".format(flash, front, front / flash))
print()
print("Cache the reusable prefix, batch the async work, tier by difficulty - stacked, ~a quarter of the standard bill.")

# Output:
# A request (reusable prefix 3500 + fresh input 500 + output 400) on frontier:
#   standard                 $0.018000
#   + prompt caching         $0.008550   (48% of standard)
#   + batch (async)          $0.009000   (50% of standard)
#   + caching AND batch      $0.004275   (24% of standard)  <- the discounts stack
#
# Model tiering - the same short query on flash vs frontier:
#   flash $0.000315  vs  frontier $0.007500  = 24x cheaper (if the small model is good enough)
#
# Cache the reusable prefix, batch the async work, tier by difficulty - stacked, ~a quarter of the standard bill.

- **Prompt caching** prices the reusable prefix at about a tenth of the input rate, dropping the request’s cost sharply.
- The **batch API** takes about half off all tokens for async work — a slower turnaround in exchange for the discount.
- The two **stack**: caching plus batch brings the request to roughly a quarter of the standard bill.
- **Model tiering** is the third lever — the same short query on flash is far cheaper than on frontier, if the small model is good enough.

#### 💡 What just happened

⚡ What just happened? Three levers cut the bill and two stack: prompt caching (reusable prefix at ~10 percent of input), the batch API (~50 percent off async), and model tiering (easy queries on a small model) - caching plus batch reaches ~25 percent of standard. Tradeoff: batch trades latency for the discount and tiering trades a little quality headroom for cost; caching is nearly free once the prefix is stable. Next: what this means at scale.

- Toggles for cached input, batch, and model tier
- Each drops the cost bar; turning caching and batch on stacks to about a quarter of standard

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Cost at Scale and Unit Economics

### Cost at Scale and Unit Economics

Project one request to a monthly bill, then to cost per user - the number that decides the business. Spend is heavily skewed (a few users drive most of it), and AI products run ~50-60 percent gross margin versus 80-90 percent for classic SaaS. Know whether your feature is even profitable.

A per-request cost is abstract; multiply it out and it gets real. Project to **daily** (times your request volume) and **monthly** (times thirty), then to the number that actually decides the business: **cost per user** (or per conversation, or per feature). Two hard truths follow. First, that number is an average that **hides a heavy skew** — a small fraction of users (or agent runs) drives a large fraction of the spend, so the mean can look fine while a few outliers quietly eat your margin. Second, **unit economics**: revenue per user minus cost per user is your gross margin, and AI products run around **fifty to sixty percent** gross margin versus **eighty to ninety percent** for classic SaaS — because inference is real cost of goods sold, not a rounding error. Without per-user cost you cannot tell whether a feature has a healthy margin or whether five heavy customers are eating the spread. The block projects and computes margin, keyless.

> ☕ **Analogy**
>
> It is the **cost per cup that decides if the cafe survives**. The owner does not care about the total coffee bean invoice; they care what a single cup costs to make against what it sells for — that per-cup margin is the whole business. And the average cup hides the regulars who camp all day refilling for free (the heavy users): a few of them can turn a profitable menu into a loss. AI cost per user is the cost per cup; the gross margin is whether the cafe stays open; and the p99 user is the customer who never leaves.

Your AI feature has a low average cost per user, but a few users use it heavily. What should you watch?

**📝 `05_scale_unit_economics.py`** - *runnable*

In [ ]:
# Project one request to a monthly bill, then to the number that decides the business: cost per user.
PRICES = {"balanced": (1.00, 5.00)}
def cost(in_tok, out_tok):
    pin, pout = PRICES["balanced"]
    return (in_tok * pin + out_tok * pout) / 1_000_000

per_req = cost(1500, 500)   # a RAG-ish request
daily_requests = 20000
monthly = per_req * daily_requests * 30
print("Project to scale (balanced model, in 1500 / out 500):")
print("  per request  ${:.6f}".format(per_req))
print("  per day      ${:>8.2f}   ({} requests)".format(per_req * daily_requests, daily_requests))
print("  per month    ${:>8.2f}".format(monthly))
print()
# Cost per user - and the number is SKEWED. A few users drive most of the spend.
users = 600                        # a heavy-usage AI product where inference is real COGS
calls_share = [0.60, 0.25, 0.15]   # top 5% of users / next 15% / everyone else
print("Cost per user is an average that HIDES the skew ({} active users):".format(users))
print("  average cost per user     ${:.4f}/month".format(monthly / users))
top5pct_users = int(users * 0.05)
print("  but the top 5% ({} users) drive {:.0f}% of the spend = ${:.2f}/month among them".format(top5pct_users, calls_share[0] * 100, monthly * calls_share[0]))
print("  -> watch the outliers (p99), not just the mean")
print()
# Gross margin: is the feature even profitable?
revenue_per_user, cost_per_user = 10.00, monthly / users
margin = (revenue_per_user - cost_per_user) / revenue_per_user * 100
print("Unit economics per user: revenue ${:.2f} - cost ${:.4f} -> gross margin {:.0f}%".format(revenue_per_user, cost_per_user, margin))
print("AI products run ~50-60% gross margin (vs 80-90% for classic SaaS) - inference is real COGS.")

# Output:
# Project to scale (balanced model, in 1500 / out 500):
#   per request  $0.004000
#   per day      $   80.00   (20000 requests)
#   per month    $ 2400.00
#
# Cost per user is an average that HIDES the skew (600 active users):
#   average cost per user     $4.0000/month
#   but the top 5% (30 users) drive 60% of the spend = $1440.00/month among them
#   -> watch the outliers (p99), not just the mean
#
# Unit economics per user: revenue $10.00 - cost $4.0000 -> gross margin 60%
# AI products run ~50-60% gross margin (vs 80-90% for classic SaaS) - inference is real COGS.

- A per-request cost projects to a **daily and monthly** bill once you multiply by volume — the abstract becomes concrete.
- **Cost per user** is the number that decides the business, but its average **hides the skew**: the top few percent of users drive most of the spend.
- **Gross margin** = (revenue per user minus cost per user) / revenue; here it lands around sixty percent.
- AI products run about **fifty to sixty percent** gross margin versus eighty to ninety for classic SaaS — inference is real cost of goods sold.

#### 💡 What just happened

⚡ What just happened? Project per-request cost to monthly and then to cost per user - the number that decides the business; spend is skewed (watch p99, not the mean), and AI gross margins run ~50-60 percent vs 80-90 percent for SaaS. Tradeoff: an AI feature can be magical and still unprofitable, so you must know its per-user cost and margin before you scale it. Next: how to attribute that cost.

- Project a per-request cost to a monthly bill
- A skewed per-user bar chart (a few users own most spend) and a gross-margin gauge

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Cost Attribution: Meter Every Call

### Cost Attribution: Meter Every Call

You cannot optimize what you cannot attribute. Tag every call with the team, user, feature, and workflow that produced it, then roll spend up by any dimension. Track median and p99 per agent-run - the median shows the normal range, the p99 exposes runaway loops.

A provider dashboard gives you one number: the total bill. It cannot tell you *which team, feature, or customer* drove it — and you cannot optimize what you cannot attribute. So you **meter every call**: each LLM call carries metadata on the same trace — the **team, user, feature, model, tokens, cost, cached flag, and an `agent_run_id`** — so you can **roll spend up by any business dimension** (by team, by feature, by customer). One number matters especially for agents: the **cost per agent-run**, tracked at both the **median and the p99**. The median shows the normal operating range; the **p99 exposes the runaway** — the loop that called a tool forty times, the run that never terminated — which a mean would smother. This metadata rides on the 12.8 trace you already emit. The block tags calls and rolls them up, keyless.

> 🧾 **Analogy**
>
> It is an **itemized receipt, not a lump-sum bill**. A restaurant bill that just says ‘four hundred dollars’ tells you nothing — you cannot tell whether it was the table’s food or one guest’s wine. An itemized receipt attributes every line to a dish and a diner, so you can see who ordered the expensive bottle (the runaway agent-run). Metering every call is printing the itemized receipt: cost meets the team, feature, and customer that ordered it, and the one outrageous line item is impossible to hide.

A provider dashboard shows a big total bill. What can it NOT tell you on its own?

**📝 `06_cost_attribution.py`** - *runnable*

In [ ]:
# You cannot optimize what you cannot attribute. Tag every call, then roll up by any dimension.
calls = [  # (team, feature, model, cost_usd, agent_run_id)
    ("product",  "chat",    "flash",    0.0003, "run-1"),
    ("product",  "chat",    "flash",    0.0004, "run-1"),
    ("product",  "summary", "frontier", 0.0180, "run-2"),
    ("research", "agent",   "frontier", 0.0090, "run-3"),
    ("research", "agent",   "frontier", 0.0110, "run-3"),
    ("research", "agent",   "frontier", 0.0120, "run-3"),
    ("research", "agent",   "frontier", 0.2400, "run-4"),   # a runaway loop
    ("product",  "chat",    "flash",    0.0003, "run-5"),
]
def rollup(key_i):
    d = {}
    for c in calls:
        d[c[key_i]] = d.get(c[key_i], 0.0) + c[3]
    return d

print("Spend rolled up by TEAM:")
for k, v in sorted(rollup(0).items(), key=lambda kv: -kv[1]):
    print("  {:<10} ${:.4f}".format(k, v))
print("Spend rolled up by FEATURE:")
for k, v in sorted(rollup(1).items(), key=lambda kv: -kv[1]):
    print("  {:<10} ${:.4f}".format(k, v))
print()
# Per agent-run cost: median shows normal, p99 exposes runaway loops.
runs = {}
for c in calls:
    runs[c[4]] = runs.get(c[4], 0.0) + c[3]
run_costs = sorted(runs.values())
def pctl(vals, p):
    k = max(0, int(round((p / 100.0) * (len(vals) - 1))))
    return sorted(vals)[k]
print("Per agent-run cost across {} runs:".format(len(run_costs)))
print("  median ${:.4f}   p99 ${:.4f}   <- p99 is {:.0f}x the median: a runaway run".format(pctl(run_costs, 50), pctl(run_costs, 99), pctl(run_costs, 99) / pctl(run_costs, 50)))
print()
print("Every call carries team / feature / model / tokens / cost / agent_run_id, so cost meets the business.")

# Output:
# Spend rolled up by TEAM:
#   research   $0.2720
#   product    $0.0190
# Spend rolled up by FEATURE:
#   agent      $0.2720
#   summary    $0.0180
#   chat       $0.0010
#
# Per agent-run cost across 5 runs:
#   median $0.0180   p99 $0.2400   <- p99 is 13x the median: a runaway run
#
# Every call carries team / feature / model / tokens / cost / agent_run_id, so cost meets the business.

- Every call carries **team / feature / model / cost / agent_run_id**, so spend can **roll up by any dimension**.
- The roll-ups by **team** and by **feature** show exactly where the money goes — a provider total never could.
- Per agent-run, the **median** is the normal range and the **p99** is far higher — a single runaway run.
- The lesson: meter every call so cost meets the business, and watch the p99 to catch the loops the mean hides.

#### 💡 What just happened

⚡ What just happened? Meter every call with team/user/feature/model/tokens/cost/agent_run_id so you can roll spend up by any dimension, and track median AND p99 per agent-run to catch runaways. Tradeoff: a little metadata plumbing on every call, in exchange for knowing exactly which team, feature, and customer drives spend. This rides on the 12.8 trace. Next: turning the meter into a dashboard that saves money.

- Tagged calls flowing into roll-ups by team and by feature
- A median-vs-p99 dial where one runaway agent-run spikes the p99

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Cost Dashboard and Optimizer

### The Cost Dashboard and Optimizer

Aggregate the metered logs into the questions that save money - spend by team and model, the cache hit rate, the expensive outliers - then name the single biggest cost driver and recommend a concrete fix. Teams that do this routinely cut LLM spend by half to two-thirds.

The metered logs from Step 6 become a **dashboard** when you aggregate them into the questions that actually save money: **spend by team and by model** (who and what burns the most), the **cache hit rate** (and what caching saved), and the **expensive outliers** (the requests above a cost threshold — usually a huge context or a frontier model on a trivial query). Then an **optimizer** names the single **biggest cost driver** and recommends a concrete fix with an estimated saving: downgrade a model tier for short queries, raise a low cache hit rate, trim a bloated context, batch the async work. The striking part is how mechanical the wins are — teams that apply this discipline routinely cut LLM spend by **half to two-thirds** in a couple of months. It is implementation discipline, not new architecture. The block builds the dashboard and the optimizer, keyless, and the illustrative wrapper shows how each call gets metered in production.

> 🔌 **Analogy**
>
> It is the **utility bill that circles your biggest drain**. A good energy report does not just show the total — it breaks usage down by appliance and points at the one that is quietly eating the bill (‘your old water heater is forty percent of this’) with a concrete fix (‘replace it, save this much’). The cost dashboard is that report for your AI app: it attributes spend by team and model, flags the outliers, and circles the single biggest driver with the change that pays for itself — usually routing trivial queries off the frontier model.

Your dashboard shows a frontier model is almost all of your spend, mostly on short queries. The best fix?

**📝 `07_dashboard_optimizer.py`** - *runnable*

In [ ]:
# Aggregate the metered logs into the questions that save money, then name the biggest driver.
logs = [  # (team, model, category, cost_usd, tokens, cached)
    ("product",  "flash",    "fast",     0.0003, 800,  True),
    ("product",  "flash",    "fast",     0.0004, 900,  False),
    ("research", "frontier", "frontier", 0.0180, 900,  False),
    ("research", "frontier", "frontier", 0.0900, 950,  False),   # frontier on a SHORT query
    ("research", "frontier", "frontier", 0.0850, 900,  False),
    ("product",  "flash",    "fast",     0.0003, 200,  True),
]
total = sum(l[3] for l in logs)
by_model = {}
for l in logs:
    by_model[l[1]] = by_model.get(l[1], 0.0) + l[3]
cache_hits = sum(1 for l in logs if l[5])

print("Cost dashboard ({} calls, ${:.4f} total):".format(len(logs), total))
print("  spend by model:")
for m, v in sorted(by_model.items(), key=lambda kv: -kv[1]):
    print("    {:<9} ${:.4f}  ({:.0f}% of spend)".format(m, v, v / total * 100))
print("  cache hit rate: {:.0f}%".format(cache_hits / len(logs) * 100))
outliers = [l for l in logs if l[3] > 0.05]
print("  expensive outliers (> $0.05): {} calls - all frontier on short queries".format(len(outliers)))
print()
# The optimizer: the biggest driver + a concrete fix with an estimated saving.
front_spend = by_model.get("frontier", 0.0)
short_frontier = [l for l in logs if l[1] == "frontier" and l[4] < 1000]
saving = sum(l[3] for l in short_frontier) * 0.85    # ~85% cheaper on flash
print("Biggest cost driver: frontier is {:.0f}% of spend, mostly on SHORT queries.".format(front_spend / total * 100))
print("  RECOMMEND: route queries under 1K tokens to flash  ->  est. saving ${:.4f} (~85% of that spend)".format(saving))
print("Discipline, not new architecture: teams routinely cut LLM spend by half to two-thirds this way.")

# Output:
# Cost dashboard (6 calls, $0.1940 total):
#   spend by model:
#     frontier  $0.1930  (99% of spend)
#     flash     $0.0010  (1% of spend)
#   cache hit rate: 33%
#   expensive outliers (> $0.05): 2 calls - all frontier on short queries
#
# Biggest cost driver: frontier is 99% of spend, mostly on SHORT queries.
#   RECOMMEND: route queries under 1K tokens to flash  ->  est. saving $0.1641 (~85% of that spend)
# Discipline, not new architecture: teams routinely cut LLM spend by half to two-thirds this way.

**📝 `cost-meter.py`** - *illustrative*

In [ ]:
# PER-CALL COST METER - an illustrative minimal example (wrap the call, read usage, tag + log).
import anthropic

client = anthropic.Anthropic()
# Illustrative prices (USD per 1M tokens); load the live sheet from a config, not hard-coded.
PRICES = {"claude-sonnet-4-6": {"in": 3.00, "out": 15.00, "cache_read": 0.30}}

def metered_chat(prompt, *, team, feature, user_id, agent_run_id):
    resp = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=512,
        messages=[{"role": "user", "content": prompt}])
    u = resp.usage                                   # the source of truth, not your guess
    p = PRICES["claude-sonnet-4-6"]
    # Anthropic reports cache reads SEPARATELY: u.input_tokens is already the fresh (non-cached) input.
    cached = getattr(u, "cache_read_input_tokens", 0)
    cost = (u.input_tokens * p["in"] + cached * p["cache_read"] + u.output_tokens * p["out"]) / 1_000_000
    # Emit a structured log so cost meets the business dimension (rolls up in your warehouse).
    log = {"team": team, "feature": feature, "user_id": user_id, "agent_run_id": agent_run_id,
           "model": "claude-sonnet-4-6", "input_tokens": u.input_tokens, "cached_tokens": cached,
           "output_tokens": u.output_tokens, "cost_usd": round(cost, 6)}
    print(log)                                       # -> Cloud Logging / OTel span attribute -> BigQuery / warehouse
    return resp.content[0].text
# Output: (illustrative minimal example - needs anthropic + a key + a warehouse sink; not run here.)

- The **dashboard** aggregates the metered logs: spend by model, the cache hit rate, and the expensive outliers (frontier on short queries).
- The **optimizer** names the single biggest driver — a frontier model on short queries — and recommends routing them to flash with an estimated saving.
- The **illustrative wrapper** is how each call is metered in production: read the API’s `usage` (including cached tokens), compute the cost, and emit a structured log tagged with team / feature / user / agent_run_id.
- Discipline, not new architecture: this is how teams routinely cut LLM spend by half to two-thirds.

#### 💡 What just happened

⚡ What just happened? Aggregate the metered logs into spend by team/model, the cache hit rate, and the outliers, then name the biggest cost driver and recommend a concrete fix - usually routing trivial queries off the frontier model. Tradeoff: none, really; this is implementation discipline that routinely cuts the bill by half to two-thirds. That is the whole cost model: see every token, attribute it, and drive it down.

- A spend dashboard by team and model with an outlier flare above a threshold
- The biggest cost driver lights up with a concrete recommendation and an estimated saving

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Knowing your AI costs is a chain from one token to the business. It starts by resetting the intuition: an LLM bills **per token, not per request**, input and output priced separately, across a huge model range (Step 1). You price **one request** exactly — and learn that a RAG call’s **retrieved context** is most of the input you pay for (Step 2). You add the **hidden multipliers** — reasoning tokens billed as invisible output, and an agent’s many calls per request — and estimate from usage, not visible text (Step 3). You cut the bill with the three levers: **cached input, batch, and model tiering**, caching plus batch stacking to about a quarter of standard (Step 4). You project to **scale and unit economics** — cost per user, the skewed distribution, and a gross margin around fifty to sixty percent (Step 5). You **attribute** every call with metadata so spend meets the team, feature, and customer, watching median and p99 (Step 6). And you build the **dashboard and optimizer** that name your biggest cost driver and its fix (Step 7). Ask four questions of any AI feature: do you know its **cost per request**, its **cost per user**, its **gross margin**, and its **single biggest cost driver**?

| The cost question | The naive assumption | The real answer |
|---|---|---|
| How am I billed? | a flat rate per request | per token, input and output apart |
| What is the input? | what the user typed | prompt + retrieved context + history |
| What is the output bill? | the visible answer | visible + hidden reasoning tokens |
| Can I cut it? | the price is fixed | cache + batch + tier (stack to ~25 percent) |
| Is the feature profitable? | the average looks fine | cost per user + margin + p99 |
| Who drove the spend? | the provider total | per-call attribution by team/feature |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now see and attribute every token. The rest of **Module 13** goes deeper on driving the bill down — model routing and distillation, quantization, and the economics of self-hosting and GPU serving. The **caching mechanics** behind this lesson’s biggest lever are in Lesson 12.4, the **trace** your cost attribution rides on is in Lesson 12.8, and org-level **cost governance and budgets** come in Module 14. The primary references are the [CloudZero LLM pricing comparison](https://www.cloudzero.com/blog/llm-api-pricing-comparison/), [Anthropic prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching) and [batch processing](https://docs.anthropic.com/en/docs/build-with-claude/batch-processing), [LiteLLM](https://github.com/BerriAI/litellm) for per-call cost tracking, and [tiktoken](https://github.com/openai/tiktoken) for counting tokens before you call.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **AI AppCosts**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-13_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-13.1.html` - regenerate this notebook whenever the source HTML is updated.*
